# Task Decomposition & Planning Architectures

This notebook demonstrates a **bounded, validated planning subsystem** for long-horizon agents. Instead of a single model looping indefinitely over intermediate thoughts, this architecture separates responsibilities:
- **Planner:** Proposes work (DAG of typed Tasks).
- **Application:** Validates constraints, budgets, and capabilities against a `GoalContract`.
- **Scheduler:** Computes topological layers and ready execution sets.
- **Replanner:** Reacts to specific evidence (e.g. source timeout) by creating a bounded `PlanPatch`.

## 1. Goal Contract and Initial Plan
We start with a strict `GoalContract` defining what is allowed and what is required. The initial plan creates a graph of tasks to research Adaptive RAG.

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('curriculum/intermediate/08-planning-task-decomposition'))
from policy import GoalContract, Task, Plan, TaskStatus, validate_plan, topological_layers

contract = GoalContract(
    goal_id="goal-research-rag",
    objective="Research adaptive RAG and produce a cited technical report.",
    audience="Technical Builders",
    required_deliverables=["report"],
    required_sections=["security implications", "routing strategies"],
    required_evidence_types=["primary_paper"],
    forbidden_actions=["execute_untrusted_code"],
    max_tasks=10,
    max_replans=2,
    max_attempts_per_task=2,
    max_total_cost_usd=2.0,
    deadline_ms=60000,
    allowed_capabilities=["search", "read_paper", "compare", "synthesize"]
)

plan_v1 = Plan(
    plan_id="plan-1",
    goal_id=contract.goal_id,
    tasks=[
        Task(task_id="t1_adaptive", task_type="read_paper", objective="Read Adaptive RAG paper.", expected_artifact_type="summary", suggested_tools=["search"]),
        Task(task_id="t2_foundation", task_type="read_paper", objective="Read RAG foundation.", expected_artifact_type="summary", suggested_tools=["search"]),
        Task(task_id="t3_implementation", task_type="read_paper", objective="Read implementation guidance.", expected_artifact_type="summary", suggested_tools=["search"]),
        Task(task_id="t4_compare", task_type="compare", objective="Compare routing strategies.", expected_artifact_type="analysis", dependencies=["t1_adaptive", "t2_foundation", "t3_implementation"], suggested_tools=["compare"]),
        Task(task_id="t5_synthesize", task_type="synthesize", objective="Synthesize report with security implications.", expected_artifact_type="report", dependencies=["t4_compare"], suggested_tools=["synthesize"])
    ]
)

# Validate against contract (throws if invalid)
validate_plan(plan_v1, contract)
print("✅ Plan V1 validated against GoalContract.")

## 2. DAG Validation and Scheduling
The scheduler groups tasks into independent topological layers that can run in parallel.

In [2]:
layers = topological_layers(plan_v1)
print("Topological Execution Schedule:")
for i, layer in enumerate(layers):
    print(f"Layer {i+1}: {[t.task_id for t in layer]}")

## 3. Simulated Execution and Failure Handling
A deterministic scheduler dispatches tasks. Notice how `t3_implementation` hits a `SOURCE_UNAVAILABLE` error, failing the task and blocking `t4_compare`.

In [3]:
from policy import TaskState, FailureCode, get_ready_tasks

states = {}

# Simulate running Layer 1
ready = get_ready_tasks(plan_v1, states)
print(f"Ready tasks: {[t.task_id for t in ready]}")

states["t1_adaptive"] = TaskState(task_id="t1_adaptive", status=TaskStatus.SUCCEEDED, artifact_id="art-1")
states["t2_foundation"] = TaskState(task_id="t2_foundation", status=TaskStatus.SUCCEEDED, artifact_id="art-2")
states["t3_implementation"] = TaskState(task_id="t3_implementation", status=TaskStatus.FAILED, error_code=FailureCode.SOURCE_UNAVAILABLE)

print("Executed Layer 1. Attempting to schedule next tasks...")
ready_now = get_ready_tasks(plan_v1, states)
print(f"Ready tasks: {[t.task_id for t in ready_now]}")
print("No tasks ready! t4_compare is blocked by t3_implementation.")

## 4. Evidence-Based Replanning via PlanPatch
Instead of "trying harder" implicitly or wiping the plan, the Replanner outputs a precise `PlanPatch`. It replaces the failed source with a new one and rewires the DAG. The failed task remains in the plan history.

In [4]:
from policy import PlanPatch, apply_plan_patch

patch = PlanPatch(
    add_tasks=[
        Task(task_id="t3_replacement", task_type="read_paper", objective="Read alternate implementation guidance.", expected_artifact_type="summary", suggested_tools=["search"])
    ],
    remove_edges=[("t3_implementation", "t4_compare")],
    add_edges=[("t3_replacement", "t4_compare")],
    reason="t3_implementation failed with SOURCE_UNAVAILABLE. Replacing with t3_replacement."
)

plan_v2 = apply_plan_patch(plan_v1, patch)
validate_plan(plan_v2, contract)

print(f"✅ Plan updated to Version {plan_v2.version} (Parent: {plan_v2.parent_version})")
print(f"Reason: {plan_v2.mutation_reason}")
print(f"t4_compare dependencies are now: {[t.dependencies for t in plan_v2.tasks if t.task_id == 't4_compare'][0]}")

## 5. Resuming Execution
With the graph repaired, execution resumes. `t3_replacement` runs, unblocking `t4_compare`, allowing the pipeline to finish.

In [5]:
# Scheduler continues with Plan V2 and existing states
ready = get_ready_tasks(plan_v2, states)
print(f"Ready tasks: {[t.task_id for t in ready]}")

states["t3_replacement"] = TaskState(task_id="t3_replacement", status=TaskStatus.SUCCEEDED, artifact_id="art-3")
ready = get_ready_tasks(plan_v2, states)
print(f"Ready tasks: {[t.task_id for t in ready]}")

states["t4_compare"] = TaskState(task_id="t4_compare", status=TaskStatus.SUCCEEDED, artifact_id="art-4")
ready = get_ready_tasks(plan_v2, states)
print(f"Ready tasks: {[t.task_id for t in ready]}")

states["t5_synthesize"] = TaskState(task_id="t5_synthesize", status=TaskStatus.SUCCEEDED, artifact_id="art-5")

print("\n🎉 Goal Complete! Execution Trace:")
for state in states.values():
    print(f" - {state.task_id}: {state.status.value}")